In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/customer_intelligence_dataset.csv"
)

df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,RevenueScore,AvgMonthlySpend,TenureGroup,Cluster,CustomerSegment
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,Yes,Electronic check,29.85,29.85,0,29.85,14.925000,New,2,Growing Customers
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,No,Mailed check,56.95,1889.50,0,1936.30,53.985714,Established,0,Loyal Customers
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,Yes,Mailed check,53.85,108.15,1,107.70,36.050000,New,2,Growing Customers
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,No,Bank transfer (automatic),42.30,1840.75,0,1903.50,40.016304,Established,0,Loyal Customers
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,Yes,Electronic check,70.70,151.65,1,141.40,50.550000,New,3,At-Risk High-Value Customers


In [2]:
import joblib

model = joblib.load(
    "../models/churn_model.pkl"
)

In [3]:
X = df.drop(
    ["Churn", "Cluster", "CustomerSegment"],
    axis=1
)

df["ChurnProbability"] = model.predict_proba(X)[:,1]

In [4]:
def assign_risk(prob):

    if prob >= 0.75:
        return "High"

    elif prob >= 0.50:
        return "Medium"

    else:
        return "Low"

In [13]:
def recommend_action(row):

    segment = row["CustomerSegment"]
    risk = row["RiskLevel"]

    recommendations = {

        ("Premium Customers", "Low"):
            "VIP rewards and premium engagement",

        ("Premium Customers", "Medium"):
            "Personalized retention offer and account review",

        ("Premium Customers", "High"):
            "Dedicated account manager and loyalty incentive",

        ("Loyal Customers", "Low"):
            "Upsell premium services",

        ("Loyal Customers", "Medium"):
            "Loyalty rewards and engagement campaign",

        ("Loyal Customers", "High"):
            "Retention outreach and contract renewal offer",

        ("Growing Customers", "Low"):
            "Monitor onboarding progress",

        ("Growing Customers", "Medium"):
            "Personalized onboarding and engagement",

        ("Growing Customers", "High"):
            "Welcome offer and proactive support outreach",

        ("At-Risk High-Value Customers", "Low"):
            "Monitor usage patterns",

        ("At-Risk High-Value Customers", "Medium"):
            "Targeted engagement campaign",

        ("At-Risk High-Value Customers", "High"):
            "Immediate retention campaign and discount"
    }

    return recommendations.get(
        (segment, risk),
        "Monitor customer behavior"
    )

In [14]:
df["RiskLevel"] = df["ChurnProbability"].apply(assign_risk)

In [15]:
df[[
    "ChurnProbability",
    "RiskLevel"
]].head()

,ChurnProbability,RiskLevel
0,0.640359,Medium
1,0.037709,Low
2,0.328103,Low
3,0.031942,Low
4,0.714762,Medium


In [16]:
df["Recommendation"] = df.apply(
    recommend_action,
    axis=1
)

In [17]:
df[[
    "CustomerSegment",
    "ChurnProbability",
    "RiskLevel",
    "Recommendation"
]].head(20)

,CustomerSegment,ChurnProbability,RiskLevel,Recommendation
0,Growing Customers,0.640359,Medium,Personalized onboarding and engagement
1,Loyal Customers,0.037709,Low,Upsell premium services
2,Growing Customers,0.328103,Low,Monitor onboarding progress
3,Loyal Customers,0.031942,Low,Upsell premium services
4,At-Risk High-Value Customers,0.714762,Medium,Targeted engagement campaign
5,At-Risk High-Value Customers,0.794082,High,Immediate retention campaign and discount
6,At-Risk High-Value Customers,0.425494,Low,Monitor usage patterns
7,Growing Customers,0.263462,Low,Monitor onboarding progress
8,At-Risk High-Value Customers,0.570372,Medium,Targeted engagement campaign
9,Loyal Customers,0.014328,Low,Upsell premium services


In [18]:
df["RiskLevel"].value_counts()

RiskLevel
Low       5585
Medium    1069
High       389
Name: count, dtype: int64

In [19]:
df.groupby(
    ["CustomerSegment", "RiskLevel"]
).size()

CustomerSegment               RiskLevel
At-Risk High-Value Customers  High          381
                              Low          1042
                              Medium        818
Growing Customers             High            8
                              Low          1575
                              Medium        170
Loyal Customers               Low          1179
Premium Customers             Low          1789
                              Medium         81
dtype: int64

In [20]:
df.to_csv(
    "../data/customer_intelligence_dataset.csv",
    index=False
)